# Imports

In [1]:
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Lasso

# Make X and Y

In [2]:
data_df = pd.read_csv('data/compound/data.csv')
data_df = data_df.sample(frac=1, random_state=67).reset_index(drop=True)
print(data_df.columns)

Index(['name', 'smiles', 'molecular_weight', 'melting_point_K',
       'boiling_point_K', 'heat_of_fusion', 'heat_of_vaporization',
       'critical_temperature', 'critical_pressure', 'flash_point', 'logP',
       'M1', 'M2', 'HM', 'H', 'mm2', 'ReZG3', 'F', 'IS', 'A', 'R', 'RR', 'ABC',
       'SC'],
      dtype='object')


In [3]:
# X: pd.DataFrame = data_df[['M1', 'M2', 'HM', 'H', 'mm2', 'ReZG3', 'F', 'IS', 'A', 'R', 'RR', 'ABC', 'SC']]
X: pd.DataFrame = data_df[['M1', 'HM', 'ReZG3', 'IS', 'SC', 'A']]
# Y: pd.DataFrame = data_df[['molecular_weight', 'melting_point_K', 'boiling_point_K', 'heat_of_fusion', 'heat_of_vaporization', 'critical_temperature', 'critical_pressure', 'flash_point']]
Y: pd.DataFrame = data_df[['melting_point_K', 'boiling_point_K']]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=67)

x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

y_scaler = StandardScaler()
Y_train = y_scaler.fit_transform(Y_train)
Y_test = y_scaler.transform(Y_test)

In [4]:
only_numeric = pd.concat([X, Y], axis=1)
only_numeric.corr()

,M1,HM,ReZG3,IS,SC,A,melting_point_K,boiling_point_K
M1,1.000000,0.985579,0.954072,0.999757,0.937674,0.981009,0.641645,0.797534
HM,0.985579,1.000000,0.990596,0.983480,0.870716,0.998696,0.646729,0.757180
ReZG3,0.954072,0.990596,1.000000,0.950704,0.805272,0.993158,0.637537,0.711933
IS,0.999757,0.983480,0.950704,1.000000,0.940704,0.979409,0.637092,0.794879
SC,0.937674,0.870716,0.805272,0.940704,1.000000,0.863458,0.572210,0.811910
A,0.981009,0.998696,0.993158,0.979409,0.863458,1.000000,0.639606,0.745151
melting_point_K,0.641645,0.646729,0.637537,0.637092,0.572210,0.639606,1.000000,0.759484
boiling_point_K,0.797534,0.757180,0.711933,0.794879,0.811910,0.745151,0.759484,1.000000


# XGBoost

In [5]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)

model.fit(X_train, Y_train)
predictions = model.predict(X_test)
mse = mean_squared_error(Y_test, predictions)
r2 = r2_score(Y_test, predictions)

print(f'r-squared: {r2}')
print(f'mse: {mse}')

r-squared: 0.3824357228577051
mse: 0.3897019374723905


# Random Forest

In [6]:

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

model.fit(X_train, Y_train)
predictions = model.predict(X_test)
mse = mean_squared_error(Y_test, predictions)
r2 = r2_score(Y_test, predictions)

print(f'r-squared: {r2}')
print(f'mse: {mse}')

r-squared: 0.41152262750156093
mse: 0.3718250799634526


# Linear Regression

In [7]:
print(Y_train)

[[ 0.18339417  0.24295889]
 [-2.09575888 -2.65616788]
 [-0.04590903  0.21446287]
 [-0.98725897 -0.77772443]
 [ 0.60579478  0.0601193 ]
 [-1.32517947 -1.38250516]
 [ 0.73553212  0.54534745]
 [ 0.88337233  0.33337407]
 [-1.41629732 -0.79139079]
 [-0.96613894 -0.62000379]
 [-1.24069934 -0.74758966]
 [ 0.44286883  0.82563451]
 [ 0.01443392 -0.57278698]
 [ 0.34028582  1.46155463]
 [-0.91484744 -0.89426318]
 [ 0.40062877 -0.14984484]
 [ 0.31252807 -0.58182849]
 [-1.0717391  -1.11846889]
 [-0.61313271 -0.89627241]
 [ 0.93164669  0.11235918]
 [ 0.59734677  2.21200064]
 [ 0.30770063  0.64480415]
 [ 0.21115192 -0.75160811]
 [ 3.55053052  2.02112414]
 [ 0.45192027  0.08222079]
 [-0.96915609 -0.94037492]
 [-0.4900331  -1.28766968]
 [-0.2993494   0.10331767]
 [-0.52141144 -0.93243848]
 [ 2.02988829  0.80403533]
 [ 0.27994288  1.85033991]
 [-0.00487582  0.40662634]
 [-0.19857668  0.15254371]
 [-0.49244682 -0.63105454]
 [ 0.75061785  1.2988073 ]
 [-0.5781338  -0.58182849]
 [ 1.48680178  0.04203626]
 

In [8]:
model = Lasso(alpha=0.01)

model.fit(X_train, Y_train)
predictions = model.predict(X_test)
mse = mean_squared_error(Y_test, predictions)
r2 = r2_score(Y_test, predictions)

print(f'r-squared: {r2}')
print(f'mse: {mse}')

r-squared: 0.4545141841970555
mse: 0.3345390769203516
